# Assignment 1 2AMM10 2023-2024

## Group: [Fill in your group name]
### Member 1: [Fill in your name]
### Member 2: [Fill in your name]
### Member 3: [Fill in your name]

In [1]:
import os
from pathlib import Path
import pandas as pd
import torch 
from torch.utils.data import Dataset
from torch import nn
from matplotlib import pyplot as plt
from PIL import Image
import numpy as np
from torchvision import transforms
import kagglehub

# add additonal imports here
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
from IPython.display import clear_output
from torchvision.models import mobilenet_v2
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
import torch.nn.functional as F
from tqdm import tqdm
import json


class FashionDataset(Dataset):
    def __init__(self, csv_file, img_dir,column_class="articleTypeId", transform=None):
        """
        Args:
            csv_file (str): Path to the CSV file with labels.
            img_dir (str): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.df = pd.read_csv(csv_file)  # load CSV file
        self.img_dir = img_dir  # image folder path
        self.transform = transform  # image transformations
        self.targets = list(self.df[column_class].values)


    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, f"{self.df.loc[idx,'imageId']}.jpg")  # Get image filename
        image = Image.open(img_name).convert("RGB")  # Load image

        if self.transform:
            image = self.transform(image)  # Apply transformations

        return image, self.targets[idx]


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/hhdel/deep-learning-assignment1/.venv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/hhdel/deep-learning-assignment1/.venv/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  F

In [4]:
print(mobilenet_v2)

<function mobilenet_v2 at 0x7373d95b6560>


Download data with kagglehub

In [3]:
dataset_path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
img_dir = os.path.join(dataset_path,"images")

The different datasets can be loaded using the class `FashionDataset` which is a custon PyTorch dataset (see [Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) for more information). Below an example of how to use the `FashionDataset` constructor as well as some visualizations. Please note that you may have to adapt the arguments to match the strucucture of your working directory.

In [4]:
BASE_DIR = Path().resolve()
DATA_PATH = BASE_DIR / "input"

print("Looking in:", DATA_PATH)


Looking in: /home/hhdel/repos/deep_learning/hw-1-deepL/input


In [5]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH / "train.csv")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["articleTypeName"])

train_df.to_csv(DATA_PATH / "train_split.csv", index=False)
val_df.to_csv(DATA_PATH / "val_split.csv", index=False)

## Loading different datasets

In [6]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.98),  # Random horizontal flip
    transforms.RandomRotation(degrees=5),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.05)], p=0.5),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# train_dataset = FashionDataset(DATA_PATH / "train.csv",img_dir,transform=transform)
train_dataset = FashionDataset(DATA_PATH / "train_split.csv",img_dir,transform=transform)
val_dataset = FashionDataset(DATA_PATH / "val_split.csv",img_dir,transform=transform)

main_test_dataset =  FashionDataset(DATA_PATH / "main_test.csv",img_dir,transform=transform)
new_test_dataset =  FashionDataset(DATA_PATH / "new_test.csv",img_dir,transform=transform)
main_support_dataset =  FashionDataset(DATA_PATH / "main_support.csv",img_dir,transform=transform)
new_support_dataset =  FashionDataset(DATA_PATH / "new_support.csv",img_dir,transform=transform)
merged_test_dataset =  FashionDataset(DATA_PATH / "merged_test.csv",img_dir,transform=transform) # merged corresponds to main+new
merged_support_dataset =  FashionDataset(DATA_PATH / "merged_support.csv",img_dir,transform=transform)

# datasets with categories
main_test_dataset_cat =  FashionDataset(DATA_PATH / "main_test.csv",img_dir,column_class="categoryId",transform=transform)
main_support_dataset_cat =  FashionDataset(DATA_PATH / "main_support.csv",img_dir,column_class="categoryId",transform=transform)

label_id_to_label_name = {i: train_dataset.df[train_dataset.df["articleTypeId"]==i]["articleTypeName"].iloc[0] for i in range(39)} 
label_id_to_label_name.update({i: new_test_dataset.df[new_test_dataset.df["articleTypeId"]==i]["articleTypeName"].iloc[0] for i in range(39,39+30)})

In [7]:
img, label = train_dataset[0]
print(f"Image size: {img.shape}")  # For PyTorch tensors, returns (C, H, W)

img, label = main_test_dataset[0]
print(f"Image size: {img.shape}")  # For PyTorch tensors, returns (C, H, W)

img, label = new_test_dataset[0]
print(f"Image size: {img.shape}")  # For PyTorch tensors, returns (C, H, W)

img, label = new_support_dataset[0]
print(f"Image size: {img.shape}")  # For PyTorch tensors, returns (C, H, W)

img, label = merged_support_dataset[0]
print(f"Image size: {img.shape}")  # For PyTorch tensors, returns (C, H, W)

img, label = main_support_dataset_cat[0]
print(f"Image size: {img.shape}")  # For PyTorch tensors, returns (C, H, W)


Image size: torch.Size([3, 224, 224])
Image size: torch.Size([3, 224, 224])
Image size: torch.Size([3, 224, 224])
Image size: torch.Size([3, 224, 224])
Image size: torch.Size([3, 224, 224])
Image size: torch.Size([3, 224, 224])


## Task 1

In [8]:
# -- Main CNN class to transfer models
class TransferLearnP2(nn.Module):
    def __init__(self, base_model, num_classes, model_name="default_model", lr=1e-3):
        super().__init__()
        self.model = base_model
        self.lr = lr
        self.num_classes = num_classes
        self.model_name = model_name
        self.criterion = nn.CrossEntropyLoss()

        # Metric tracking
        self.train_losses, self.val_losses = [], []
        self.train_accs, self.val_accs = [], []
        self.train_aucs, self.val_aucs = [], []
        self.train_balanced_accs, self.val_balanced_accs = [], []

    def forward(self, x):
        return self.model(x)

    # -- Optimizer
    def get_optimizer(self, lr=None):
        lr = lr if lr is not None else self.lr
        if lr is None:
            raise ValueError("Learning rate is not defined")
        return torch.optim.RMSprop(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=lr
        )


    def train_one_epoch(self, dataloader, optimizer, device):
        self.train()
        epoch_loss = 0.0
        correct = 0
        total = 0
        probs_all = []
        targets_all = []

        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            logits = self(x)
            loss = self.criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            probs_all.append(probs.detach().cpu())
            targets_all.append(y.detach().cpu())

        # Flatten predictions and targets
        all_probs = torch.cat(probs_all)
        all_targets = torch.cat(targets_all)
        all_preds = all_probs.argmax(dim=1)

        avg_loss = epoch_loss / total
        acc = correct / total

        # Balanced accuracy
        balanced_acc = balanced_accuracy_score(all_targets.numpy(), all_preds.numpy())

        # AUC (optional, handle exception)
        try:
            auc = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro')
        except Exception as e:
            print(f"Could not compute train AUC: {e}")
            auc = float('nan')

        # Store metrics
        self.train_losses.append(avg_loss)
        self.train_accs.append(acc)
        self.train_balanced_accs.append(balanced_acc)  # NEW: store balanced accuracy
        self.train_aucs.append(auc)

        return avg_loss, acc, balanced_acc, auc


    def validate_one_epoch(self, dataloader, device):
        self.eval()
        epoch_loss = 0.0
        correct = 0
        total = 0
        probs_all = []
        targets_all = []

        with torch.no_grad():
            for x, y in dataloader:
                x, y = x.to(device), y.to(device)
                logits = self(x)
                loss = self.criterion(logits, y)

                epoch_loss += loss.item() * x.size(0)
                probs = F.softmax(logits, dim=1)
                preds = probs.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)

                probs_all.append(probs.cpu())
                targets_all.append(y.cpu())

        # Flatten predictions and targets
        all_probs = torch.cat(probs_all)
        all_targets = torch.cat(targets_all)
        all_preds = all_probs.argmax(dim=1)

        avg_loss = epoch_loss / total
        acc = correct / total

        # Balanced accuracy
        balanced_acc = balanced_accuracy_score(all_targets.numpy(), all_preds.numpy())

        # AUC (optional, handle exception)
        try:
            auc = roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro')
        except Exception as e:
            print(f"Could not compute val AUC: {e}")
            auc = float('nan')

        # Store metrics
        self.val_losses.append(avg_loss)
        self.val_accs.append(acc)
        self.val_balanced_accs.append(balanced_acc)  # NEW: store balanced accuracy
        self.val_aucs.append(auc)

        return avg_loss, acc, balanced_acc, auc
    

    def plot_metrics(self, save_dir="./output", filename="final_metrics.png", test_loss=None, test_acc=None, test_bal_acc=None):
        clear_output(wait=True)
        epochs = range(1, len(self.train_losses) + 1)
        plt.figure(figsize=(16, 5))

        # -- Use nicer colors
        train_color = "#1f77b4"  # blue
        val_color = "#ff7f0e"    # orange
        test_color = "#2ca02c"   # green

        # -- Loss plot
        plt.subplot(1, 3, 1)
        plt.plot(epochs, self.train_losses, label='Train Loss', color=train_color)
        if len(self.val_losses) > 0:
            plt.plot(epochs, self.val_losses, label='Val Loss', color=val_color)
        if test_loss is not None:
            plt.scatter(epochs[-1], test_loss, color=test_color, s=80, marker='*', label='Test Loss', edgecolors='black', zorder=5)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()

        # -- Accuracy plot
        plt.subplot(1, 3, 2)
        plt.plot(epochs, self.train_accs, label='Train Acc', color=train_color)
        if len(self.val_accs) > 0:
            plt.plot(epochs, self.val_accs, label='Val Acc', color=val_color)
        if test_acc is not None:
            plt.scatter(epochs[-1], test_acc, color=test_color, s=80, marker='*', label='Test Acc', edgecolors='black', zorder=5)
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()

        # -- Balanced Accuracy plot
        plt.subplot(1, 3, 3)
        plt.plot(epochs, self.train_balanced_accs, label='Train Bal Acc', color=train_color)
        if len(self.val_balanced_accs) > 0:
            plt.plot(epochs, self.val_balanced_accs, label='Val Bal Acc', color=val_color)
        if test_bal_acc is not None:
            plt.scatter(epochs[-1], test_bal_acc, color=test_color, s=80, marker='*', label='Test Bal Acc', edgecolors='black', zorder=5)
        plt.xlabel('Epoch')
        plt.ylabel('Balanced Accuracy')
        plt.legend()

        # -- Save final plot
        save_dir_ = Path(save_dir) / self.model_name
        save_dir_.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_dir_ / filename, bbox_inches='tight')

        plt.show()


class CNET_HHD0_V44(nn.Module):
    def __init__(self, num_classes=315, input_shape=(3, 224, 224), class_weights=None, in_ch=256, drop_1=0.0, drop_2=0.0):
        super().__init__()
        self.in_ch = in_ch
        self.drop_1 = drop_1
        self.drop_2 = drop_2
        self.input_shape = input_shape

        self.class_weights = None
        
        # -- Feature extractor blocks
        self.block1 = nn.Sequential(
            nn.Conv2d(3, in_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_ch),
            nn.GELU(),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(drop_1+0.05)
        )
        
        self.block2 = nn.Sequential(
            nn.Conv2d(in_ch, in_ch*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_ch*2),
            nn.GELU(),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(drop_1+0.05)
        )
        
        self.block3 = nn.Sequential(
            nn.Conv2d(in_ch*2, in_ch*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_ch*4),
            nn.GELU(),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(drop_1+0.1)
        )        

        self.block4 = nn.Sequential(
            nn.Conv2d(in_ch*4, in_ch*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_ch*8),
            nn.GELU(),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(drop_1+0.15)
        )
       
        self.block5 = nn.Sequential(
            nn.Conv2d(in_ch*8, in_ch*16, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_ch*16),
            nn.GELU(),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(drop_1+0.2),
        )

        self.block6 = nn.Sequential(
            nn.Conv2d(in_ch*16, in_ch*32, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_ch*32),
            nn.GELU(),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(drop_1+0.2),
        )        

        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            x = self.block1(dummy)
            x = self.block2(x)
            x = self.block3(x)
            x = self.block4(x)
            x = self.block5(x)
            x = self.block6(x)
            self._flattened_size = x.view(1, -1).size(1) 

        
        # -- Classifier block
        self.classifier = nn.Sequential(
            nn.Linear(self._flattened_size, in_ch*16),     # fc1
            nn.LayerNorm(in_ch*16),                      # ln1
            nn.ELU(),
            nn.Dropout(drop_2),
            nn.Linear(in_ch*16, num_classes),       # fc2
            nn.BatchNorm1d(num_classes)            # ln2
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = x.view(x.size(0), -1)  
        x = self.classifier(x)
        probs = F.softmax(x, dim=1)

        return probs

In [9]:
# -- Train the transfered models
def train_transfer_model(config, model_name, train_loader, val_loader, test_loader, epochs=10, save_dir="./output"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(save_dir, exist_ok=True)

    # -- Step 6.1 Get number of classes. It is not streamlined for other applications
    num_classes = 39  

    # -- Step 6.2 Build base model using config
    base_model = config["model_fn"](num_classes)

    # -- Step 6.3 Wrap it in TransferLearnP2 wrapper
    trainer = TransferLearnP2(base_model, num_classes=num_classes, model_name=model_name, lr=config["lr"])
    trainer.to(device)

    # -- Step 6.4 Set Optimizer (optimizer is defined within the TransferLearnP2 class). Could have been tuned...
    optimizer = trainer.get_optimizer()

    # -- Step 6.5 Print the layers to be trained
    n_trainable = sum(p.numel() for p in trainer.parameters() if p.requires_grad)
    print(f"Trainable parameters in [{trainer}]: {n_trainable:,}")

    # -- Step 6.6 Training loop
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc, train_bal_acc, train_auc = trainer.train_one_epoch(train_loader, optimizer=optimizer, device=device)
        val_loss, val_acc, val_bal_acc, val_auc = trainer.validate_one_epoch(val_loader, device=device)

        print(f"\nEpoch {epoch+1}/{epochs}")
        # Live plot
        # trainer.plot_metrics(save_path=f"epoch_{epoch+1}.png")

    # -- Step 6.7 Test set evaluation
    trainer.to(device)
    trainer.eval()
    test_loss = 0.0
    correct, total = 0, 0
    all_preds = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = trainer(X_batch)
            loss = trainer.criterion(outputs, y_batch)

            # -- Statistics
            test_loss += loss.item() * X_batch.size(0)
            _, predicted = outputs.max(1)  # no softmax
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

            # For AUC, we still need softmax probabilities
            probs = F.softmax(outputs, dim=1)

            all_preds.append(predicted.cpu())
            all_targets.append(y_batch.cpu())
            all_probs.append(probs.cpu())

    # Combine all batches
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    all_probs = torch.cat(all_probs)

    # Compute average loss and accuracy
    test_loss /= total
    test_acc = correct / total

    # Compute balanced accuracy
    test_balanced_acc = balanced_accuracy_score(all_targets.numpy(), all_preds.numpy())

    # Compute AUC
    try:
        auc = roc_auc_score(all_targets.numpy(), all_probs.numpy(), multi_class='ovr', average='macro')
    except Exception as e:
        print(f"Could not compute test AUC: {e}")
        auc = float('nan')

    # Report
    print(f"Test → Loss: {test_loss:.4f}, Acc: {test_acc:.4f}, Balanced Acc: {test_balanced_acc:.4f}, AUC: {auc:.4f}")


    # After all epochs finished and test evaluation done
    trainer.plot_metrics(
        save_dir="./output",
        filename="final_metrics.png",
        test_loss=test_loss,
        test_acc=test_acc,
        test_bal_acc=test_balanced_acc
    )

    # -- Step 6.8 Save metrics
    metrics = {
        "train_loss": trainer.train_losses,
        "train_auc": trainer.train_aucs,
        "train_acc": trainer.train_accs,
        "train_bal_acc": trainer.train_balanced_accs,
        
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_bal_acc": test_balanced_acc,
    
        "val_loss": trainer.val_losses,
        "val_auc": trainer.val_aucs,
        "val_acc": trainer.val_accs,
        "val_bal_acc": trainer.val_balanced_accs,
    }

    # -- Step 6.9 Save model
    torch.save(trainer.model.state_dict(), os.path.join(save_dir, f"{model_name}.pth"))
    print(f"Model weights saved to: {save_dir}")
    
    # -- Step 6.10: Save metrics
    with open(os.path.join(save_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"\nMetrics saved to {save_dir}/metrics.json")



# -- Build models
def build_mobilenet_v2(model_import=None, num_classes=39, weights=None):
    model = model_import()
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def build_HHD_model(num_classes=39, in_ch=64, drop_1=0.0, drop_2=0.0):
    model = CNET_HHD0_V44(num_classes=num_classes, in_ch=in_ch, drop_1=drop_1, drop_2=drop_2)
    return model


def configure_finetuning(model):
    for param in model.parameters():
        param.requires_grad = True
    
    # Debug print
    print("Trainable Parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"   • {name}")

    return model

# -- Add the transformers to the dataloaders streamlined
def build_dataloaders(train, val, main_test, batch_size=32):
    train_loader = DataLoader(train, batch_size=32, shuffle=True,
                                num_workers=4,
                                pin_memory=True,
                                persistent_workers=True)
    
     # Validation data loader. Leave this unchanged.
    val_loader = DataLoader(val, batch_size=32, shuffle=False,
                                num_workers=4,
                                pin_memory=True,
                                persistent_workers=True)   
    
    # Test data loader. Leave this unchanged.
    test_loader = DataLoader(main_test, batch_size=32, shuffle=False,
                                num_workers=4,
                                pin_memory=True,
                                persistent_workers=True)

    return train_loader, val_loader, test_loader


In [10]:
df = pd.read_csv(DATA_PATH / "train.csv")
df2 = pd.read_csv(DATA_PATH / "main_test.csv")
df2

,imageId,articleTypeName,categoryName,articleTypeId,categoryId
0,59443,Shirts,Topwear,1,0
1,6073,Tops,Topwear,6,0
2,47346,Clutches,Bags,31,2
3,20793,Jackets,Topwear,35,0
4,32585,Trousers,Bottomwear,21,4
...,...,...,...,...,...
1992,22180,Casual Shoes,Shoes,2,1
1993,4547,Sports Shoes,Shoes,4,1
1994,49919,Sarees,Saree,25,16
1995,8774,Shirts,Topwear,1,0


In [11]:
# -- Build a configuration dictionary of the main models
model_configs = {
    "transfer_learn_1": {
    "model_fn": build_HHD_model,
    "lr": 0.0001,
    },

    "transfer_learn_2": {
        "model_fn": mobilenet_v2,
        "lr": 0.000005,
    },    
}

not_training = True
# -- The training process is described step wise, follow the numbers

if not_training:
    pass
else:
    # -- 0 Select directory to dump files
    save_dir = "./output"

    # -- 0 Logic begins here
    for name, config in model_configs.items():
        print(f"\nTraining | {name} | {config['model_fn'].__name__}")
        # -- 1 Load the Data loaders with these transformations
        train_loader, val_loader, test_loader = build_dataloaders(
            train = train_dataset,
            val =  val_dataset,
            main_test = main_test_dataset
        )
        
        # -- 2 Defines the model
        print(f"Setting up model: {name}")
        model = config["model_fn"](num_classes=39)

        # -- 3 Save path
        model_name = f"{config['model_fn'].__name__}"
        save_path = os.path.join(save_dir, model_name)
        os.makedirs(save_path, exist_ok=True)

        # -- 4 Train model. You can go to this function to check what it does...
        train_transfer_model(
            config=config,
            model_name=model_name,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            epochs=25,
            save_dir=save_path
        )


In [12]:
mobilenet = mobilenet_v2(num_classes=39)
print(mobilenet)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [ ]:
# your solution
# train dataset: train_dataset
# test datset: main_test_dataset

## Task 2

In [ ]:
# your solution
# train dataset: train_dataset
# test and support dataset: see scenarios table

## Task 3

In [ ]:
# your solution
# test dataset: merged_test_dataset
# support/catalog dataset: support_test_dataset

## Task 4

In [ ]:
# your solution
# datasets: first 10 classes of train_dataset and main_test_dataset